In [13]:
# =============================================================================
# INDIA TODAY FACT CHECK SCRAPER (2022–2025)

# =============================================================================

# -----------------------------
# 1. INSTALL REQUIRED PACKAGES
# -----------------------------
%pip -q install google-colab-selenium beautifulsoup4 pandas requests tqdm python-dateutil


In [ ]:
# -----------------------------
# 2. IMPORT LIBRARIES
# -----------------------------
import json
import random
import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
import google_colab_selenium as gs
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    StaleElementReferenceException,
    ElementClickInterceptedException,
    TimeoutException,
    WebDriverException,
)

from google.colab import drive

In [ ]:
# -----------------------------
# 3. MOUNT GOOGLE DRIVE
# -----------------------------
drive.mount("/content/drive")

# Change this folder if you want the project stored elsewhere in Drive.
DRIVE_FOLDER = Path("/content/drive/MyDrive/IndiaToday_FactCheck_Scraper")
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

# Files saved in Drive:
URLS_CSV = DRIVE_FOLDER / "indiatoday_factcheck_urls.csv"
PROGRESS_CSV = DRIVE_FOLDER / "indiatoday_factcheck_progress_all.csv"
ALL_CSV = DRIVE_FOLDER / "indiatoday_factcheck_all.csv"
FILTERED_CSV = DRIVE_FOLDER / "indiatoday_factcheck_2022_2025.csv"

In [ ]:
# -----------------------------
# 4. SCRAPER SETTINGS
# -----------------------------
BASE_URL = "https://www.indiatoday.in"
START_URL = "https://www.indiatoday.in/fact-check"

START_YEAR = 2022
END_YEAR = 2025

# Maximum number of Load More clicks during a fresh crawl.
MAX_CLICKS = 300

# Save the URL list after every N clicks, and article progress after every
# N attempted article pages.
URL_CHECKPOINT_EVERY = 5
ARTICLE_CHECKPOINT_EVERY = 10

# Waits and delays (seconds).
INITIAL_PAGE_WAIT = 5
LOAD_MORE_WAIT = 4
SCROLL_WAIT = 1.5
REQUEST_DELAY = (0.4, 0.8)

In [ ]:
# -----------------------------
# 5. COMMON HELPERS
# -----------------------------
def clean_text(text):
    """Normalize whitespace and return None for empty strings."""
    if text is None:
        return None
    text = re.sub(r"\xa0", " ", str(text))
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    text = text.strip()
    return text or None


def save_csv_safely(dataframe, path):
    """Write to a temporary file before replacing the destination file."""
    path = Path(path)
    temp_path = Path(str(path) + ".tmp")
    dataframe.to_csv(temp_path, index=False, encoding="utf-8-sig")
    temp_path.replace(path)


def normalize_article_url(href):
    """Convert an href to a canonical India Today Fact Check story URL."""
    if not href:
        return None

    full_url = urljoin(BASE_URL, href.strip())
    parsed = urlparse(full_url)

    if parsed.netloc.lower() not in {"www.indiatoday.in", "indiatoday.in"}:
        return None

    if "/fact-check/story/" not in parsed.path.lower():
        return None

    # Discard query strings and fragments to reduce duplicate URLs.
    return f"{BASE_URL}{parsed.path}".rstrip("/")

In [ ]:
# -----------------------------
# 6. SELENIUM: FIND AND COLLECT STORY LINKS
# -----------------------------
driver = gs.Chrome()
wait = WebDriverWait(driver, 20)

article_links = set()


def collect_links():
    """Collect Fact Check story URLs currently present in the rendered page."""
    soup = BeautifulSoup(driver.page_source, "html.parser")
    newly_found = 0

    for anchor in soup.select("a[href]"):
        article_url = normalize_article_url(anchor.get("href"))
        if article_url and article_url not in article_links:
            article_links.add(article_url)
            newly_found += 1

    return newly_found


def find_load_more():
    """Locate a visible Load More button, trying the known class first."""
    candidates = driver.find_elements(By.CSS_SELECTOR, "div.viewall__btn")

    for element in candidates:
        try:
            if element.is_displayed() and "load more" in element.text.lower():
                return element
        except StaleElementReferenceException:
            continue

    # Fallback: match an element whose normalized visible text is 'load more'.
    candidates = driver.find_elements(
        By.XPATH,
        "//*[normalize-space(translate(.,"
        "'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'))='load more']",
    )

    for element in candidates:
        try:
            if element.is_displayed():
                return element
        except StaleElementReferenceException:
            continue

    return None


def save_url_checkpoint():
    """Persist the URLs collected so far."""
    urls_df = pd.DataFrame({"url": sorted(article_links)})
    save_csv_safely(urls_df, URLS_CSV)
    print(f"URL checkpoint saved: {len(urls_df)} URLs")

In [ ]:
# -----------------------------
# 7. CRAWL THE LISTING PAGE
# -----------------------------
# If a URL checkpoint exists, load it and continue collecting from the listing.
if URLS_CSV.exists():
    existing_urls_df = pd.read_csv(URLS_CSV)
    if "url" in existing_urls_df.columns:
        article_links.update(
            existing_urls_df["url"].dropna().astype(str).map(str.strip)
        )
    print(f"Loaded {len(article_links)} URLs from the existing Drive checkpoint.")
else:
    print("No existing URL checkpoint found; starting a fresh URL crawl.")

try:
    driver.get(START_URL)
    time.sleep(INITIAL_PAGE_WAIT)

    collect_links()
    print("URLs detected after initial page load:", len(article_links))
    save_url_checkpoint()

    clicks_since_checkpoint = 0

    for click_number in range(1, MAX_CLICKS + 1):
        collect_links()
        old_count = len(article_links)

        load_more = find_load_more()

        if load_more is None:
            # Scroll to the bottom and check once more for lazy-loaded controls.
            driver.execute_script(
                "window.scrollTo(0, document.body.scrollHeight);"
            )
            time.sleep(SCROLL_WAIT)
            load_more = find_load_more()

        if load_more is None:
            print("Load More control not found. Ending archive crawl.")
            break

        try:
            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                load_more,
            )
            time.sleep(0.8)

            try:
                load_more.click()
            except (
                ElementClickInterceptedException,
                StaleElementReferenceException,
            ):
                # Re-find the control in case the page re-rendered.
                load_more = find_load_more()
                if load_more is None:
                    print("Load More control disappeared before clicking.")
                    continue
                driver.execute_script("arguments[0].click();", load_more)

            # Allow the next batch to load and render.
            time.sleep(LOAD_MORE_WAIT)
            driver.execute_script(
                "window.scrollTo(0, document.body.scrollHeight);"
            )
            time.sleep(SCROLL_WAIT)

            collect_links()
            new_count = len(article_links) - old_count

            print(
                f"Click {click_number}: {len(article_links)} total URLs; "
                f"{new_count} newly detected"
            )

            clicks_since_checkpoint += 1
            if clicks_since_checkpoint >= URL_CHECKPOINT_EVERY:
                save_url_checkpoint()
                clicks_since_checkpoint = 0

            # If a click yields no new links, continue trying: the page may
            # need another interaction. The loop ends at the click limit or
            # when the Load More control disappears.

        except WebDriverException as exc:
            print(f"Browser error on click {click_number}: {exc}")
            save_url_checkpoint()
            time.sleep(2)

    # Save all URLs after the crawl, even if it ended early.
    save_url_checkpoint()

except Exception as exc:
    # Preserve the URLs collected so far if an unexpected crawling error occurs.
    save_url_checkpoint()
    print(f"Unexpected listing-crawl error: {repr(exc)}")

finally:
    # The browser is no longer needed for article extraction.
    try:
        driver.quit()
    except Exception:
        pass

article_links = sorted(article_links)
print("\nTotal unique article URLs collected:", len(article_links))


In [ ]:

# -----------------------------
# 8. REQUEST SESSION FOR ARTICLE EXTRACTION
# -----------------------------
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
})

In [ ]:

# -----------------------------
# 9. ARTICLE EXTRACTION FUNCTION
# -----------------------------
def extract_article(url, retries=3, timeout=30):
    """Extract title, publication date, and article text from a story URL."""
    result = {
        "url": url,
        "title": None,
        "date": None,
        "article_text": None,
        "status": None,
    }

    response = None

    for attempt in range(1, retries + 1):
        try:
            response = session.get(url, timeout=timeout)
            result["status"] = response.status_code
            response.raise_for_status()
            break
        except requests.RequestException as exc:
            result["status"] = f"Request error: {exc}"
            if attempt < retries:
                time.sleep(attempt * 2)
            else:
                return result

    soup = BeautifulSoup(response.text, "html.parser")

    # TITLE: Open Graph metadata, then H1, then document title.
    og_title = soup.find("meta", property="og:title")
    if og_title and og_title.get("content"):
        result["title"] = clean_text(og_title.get("content"))
    elif soup.find("h1"):
        result["title"] = clean_text(soup.find("h1").get_text(" ", strip=True))
    elif soup.title:
        result["title"] = clean_text(soup.title.get_text(" ", strip=True))

    # PUBLICATION DATE: check common metadata fields first.
    date_metadata = [
        ("meta", {"property": "article:published_time"}),
        ("meta", {"name": "publish-date"}),
        ("meta", {"name": "date"}),
        ("meta", {"name": "DC.date.issued"}),
        ("meta", {"itemprop": "datePublished"}),
    ]

    for tag, attrs in date_metadata:
        element = soup.find(tag, attrs)
        if element:
            date_value = (
                element.get("content")
                or element.get("datetime")
                or element.get_text(" ", strip=True)
            )
            if date_value:
                result["date"] = clean_text(date_value)
                break

    # DATE FALLBACK: inspect JSON-LD structured data, including @graph objects.
    if not result["date"]:
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                raw_data = script.string or script.get_text()
                data = json.loads(raw_data)
            except (json.JSONDecodeError, TypeError):
                continue

            objects_to_check = data if isinstance(data, list) else [data]

            while objects_to_check:
                obj = objects_to_check.pop(0)
                if not isinstance(obj, dict):
                    continue

                graph = obj.get("@graph")
                if isinstance(graph, list):
                    objects_to_check.extend(graph)

                date_value = obj.get("datePublished") or obj.get("dateCreated")
                if date_value:
                    result["date"] = str(date_value).strip()
                    break

            if result["date"]:
                break

    # ARTICLE BODY: use likely story containers, then fall back to the page body.
    content_selectors = [
        "div.story__content",
        "div.story-content",
        "div.content-area",
        "div.article-body",
        "div[itemprop='articleBody']",
        "article",
    ]

    article_container = None
    for selector in content_selectors:
        candidate = soup.select_one(selector)
        if candidate and len(candidate.get_text(" ", strip=True)) > 200:
            article_container = candidate
            break

    if article_container is None:
        article_container = soup.body or soup

    # Remove scripts, embeds, and common non-article elements before extraction.
    for unwanted in article_container.select(
        "script, style, noscript, iframe, "
        "figure, .advertisement, .ad, .social-share"
    ):
        unwanted.decompose()

    paragraphs = [
        clean_text(paragraph.get_text(" ", strip=True))
        for paragraph in article_container.find_all("p")
    ]
    paragraphs = [paragraph for paragraph in paragraphs if paragraph and len(paragraph) > 30]

    text = "\n".join(paragraphs).strip()

    # Fallback when paragraph-based extraction returns very little text.
    if len(text) < 100:
        text = clean_text(article_container.get_text(" ", strip=True))

    result["article_text"] = text or None

    # Status identifies successful extraction and common missing-data cases.
    if not result["title"] and not result["date"] and not result["article_text"]:
        result["status"] = "no_fields_extracted"
    elif not result["article_text"]:
        result["status"] = "content_missing"
    elif not result["title"]:
        result["status"] = "title_missing"
    else:
        result["status"] = "success"

    return result


In [ ]:
# -----------------------------
# 10. LOAD ARTICLE CHECKPOINT AND RESUME
# -----------------------------
columns = ["url", "title", "date", "article_text", "status"]

if PROGRESS_CSV.exists():
    progress_df = pd.read_csv(PROGRESS_CSV).fillna("")

    # Add any columns absent from an older checkpoint.
    for column in columns:
        if column not in progress_df.columns:
            progress_df[column] = ""

    progress_df = progress_df[columns]
    progress_df = progress_df.drop_duplicates(subset="url", keep="last")
    print(f"Loaded {len(progress_df)} article records from Drive.")
else:
    progress_df = pd.DataFrame(columns=columns)
    print("No article progress checkpoint found; starting extraction.")

processed_urls = set(progress_df["url"].astype(str))
remaining_urls = [url for url in article_links if url not in processed_urls]

print("URLs already attempted:", len(processed_urls))
print("URLs remaining:", len(remaining_urls))


In [ ]:
# -----------------------------
# 11. EXTRACT ARTICLES AND SAVE PERIODIC CHECKPOINTS
# -----------------------------
new_records = []

for count, url in enumerate(
    tqdm(remaining_urls, desc="Extracting India Today articles"),
    start=1,
):
    record = extract_article(url)
    new_records.append(record)

    # Save every configured number of attempted URLs.
    if len(new_records) >= ARTICLE_CHECKPOINT_EVERY:
        progress_df = pd.concat(
            [progress_df, pd.DataFrame(new_records, columns=columns)],
            ignore_index=True,
        )
        progress_df = progress_df.drop_duplicates(subset="url", keep="last")
        save_csv_safely(progress_df, PROGRESS_CSV)
        new_records = []
        print(f"Article checkpoint saved: {len(progress_df)} records.")

    time.sleep(random.uniform(*REQUEST_DELAY))

# Save any records remaining after the loop.
if new_records:
    progress_df = pd.concat(
        [progress_df, pd.DataFrame(new_records, columns=columns)],
        ignore_index=True,
    )
    progress_df = progress_df.drop_duplicates(subset="url", keep="last")

save_csv_safely(progress_df, PROGRESS_CSV)
print(f"Full article checkpoint saved: {PROGRESS_CSV}")


In [ ]:
# -----------------------------
# 12. SAVE COMPLETE AND YEAR-FILTERED DATASETS
# -----------------------------
# Parse dates to UTC for consistent year extraction. Invalid dates become NaT.
progress_df["parsed_date"] = pd.to_datetime(
    progress_df["date"], errors="coerce", utc=True
)
progress_df["year"] = progress_df["parsed_date"].dt.year

# Save the complete set of attempted article records.
save_csv_safely(progress_df, ALL_CSV)

# Filter to articles with publication years from START_YEAR through END_YEAR.
df_2022_2025 = progress_df[
    progress_df["year"].between(START_YEAR, END_YEAR)
].copy()

df_2022_2025 = df_2022_2025.drop_duplicates(subset="url", keep="last")
save_csv_safely(df_2022_2025, FILTERED_CSV)

print("\n" + "=" * 65)
print("SCRAPING SUMMARY")
print("=" * 65)
print("Unique article URLs collected:", len(article_links))
print("Article records attempted:", len(progress_df))
print(f"Articles dated {START_YEAR}–{END_YEAR}:", len(df_2022_2025))
print("Titles extracted:", df_2022_2025["title"].replace("", pd.NA).notna().sum())
print("Dates extracted:", df_2022_2025["date"].replace("", pd.NA).notna().sum())
print("Article texts extracted:", df_2022_2025["article_text"].replace("", pd.NA).notna().sum())
print("\nArticles by year:")
print(df_2022_2025["year"].value_counts().sort_index())
print("\nMissing article text:", df_2022_2025["article_text"].isna().sum())
print("\nSaved files:")
print("URL checkpoint:", URLS_CSV)
print("Article progress:", PROGRESS_CSV)
print("All records:", ALL_CSV)
print("2022–2025 dataset:", FILTERED_CSV)
print("=" * 65)

display(df_2022_2025.head(10))

# All outputs are saved in Google Drive; no browser download is required.
